## Hierarchical agent team structure :
### 1-Overall supervisor agent
### 2-Research team
### 3-Document authoring team

#### To make our example practical let's simulate Financial Audit Team ,This team is responsible for gathering financial data, performing analysis, and generating a structure a structured audit report we'll break this process down into three main components

## A)Building the Research Team:
#### 1-Data collector Agent
#### 2-Transaction Verifier Agent

In [ ]:
from langchain_core.tools import tool


@tool
def verify_transactions(transaction_ids: list) -> list:
    """Verify each transaction ID for accuracy"""
    # Simulate verification process
    verify_transactions = [tx_id for tx_id in transaction_ids if tx_id.is_valid()]
    return verify_transactions

### Defining the Research Team Graph

In [ ]:
from langgraph.graph import StateGraph

# Define the Research Team Graph 
research_team_Graph = StateGraph()
research_team_Graph.add_node("DataCollector", #data_collector_noce)
research_team_Graph.add_node("TransactionVerifier", #transaction_verifier_node)

#Define edges to route takes between agents and back to the team supervisior
research_team_Graph.add_edge("DataCollector","TransactionVerifier")
research_team_Graph.add_edge("TransactionVerifier", "ResearchTeamSupervisor")

### Testing the Research Team

In [ ]:
# Initial task to start the reseaarch team 
initial_state = {"company_id": "1234", "period": "Q4"}
research_result = research_team_Graph.invoke(initial_state)
print(research_result)

## B)Building the Document Authoring Team:
#### 1-Financial Analyst Agent
#### 2-Report Writer Agent
#### 3-Chart Generator Agent

In [ ]:
@tool
def perform_financial_analysis(data: dict) -> dict:
    """Calculate the financial metrics from the provided data."""
    results = {
        "debt_to_equity_ratio": data["total_debt"] / data["total_equity"],
        "current_ratio": data["current_assests"] / data["current_liabilities"]
    }
    return results

In [ ]:
from pathlib import Path

@tool
def writer_report(content: str, file_name: str) -> str:
    """Write the aduit to a text file."""
    report_path = Path(f"/report/{file_name}")
    report_path.write_text(content)
    return f"Report saved at {report_path}"

In [ ]:
from langchain_experimental.utilities import PythonREPL
repl = PythonREPL()

@tool
def generaet_chart(data: dict, chart_type: str) -> str:
    """Generate a chart for financial data"""
    code = f"""
import matplotlib.pyplot as plt
data = {data}
plt.figure(figsize=(10,6))
if "{chart_type}" == "bar":
    plt.bar(data.keys(), data.values())
elif "{chart_type}" == "line":
    plt.bar(data.keys(), data.values())
plt.title('Financial Metrics')
"""
    repl.run(code)
    return "Chart generated and saved at /charts/financial_chart.png"

### Defining the Document Authoring Team Graph

In [ ]:
# Define the document Authoring Team Graph
document_authoring_graph = StateGraph()
document_authoring_graph.add_node("FinancialAnalyst", financial_analyst_node)
document_authoring_graph.add_node("ReportWriter", report_writer_node)
document_authoring_graph.add_node("ChartGenerator", Chart_generator_node)

# Define edges to route tasks 
document_authoring_graph.add_edge("FinancialAnalyst","ReportWriter")
document_authoring_graph.add_edge("ReportWriter", "ChartGenerator")
document_authoring_graph.add_edge("ChartGenerator","DocumentAuthoringSupervisor")

                                  

### Testing the Document Authoring Team

In [ ]:
sample_data = {
    "total_debt": 10000000,
    "total_equity": 50000,
    "current_assets": 8000,
    "current_liabilities": 4000
}

authoring_results = document_authoring_graph.invoke(sample_data)
print(authoring_results)

## C)Integrating the Teams with an overall supervisor

In [ ]:
# Define the overall supervisor node
overall_supervisor = creat_team_supervisor(
    llm,
    "Manage financial audit workflow bwtween data collection and report generation teams",
    ["ResearchTeam", "DocumentAuthoringTeam"]
)
# Define the top level graph for the financial audit team
financial_audit_garph = StateGraph()
financial_audit_garph.add_node("ResearchTeam", research_team_Graph)
financial_audit_garph.add_node("DocumentAuthoringTeam", document_authoring_graph)
financial_audit_garph.add_node("OverallSupervisor", overall_supervisor)

# Define routing through the overall supervisor
financial_audit_garph.add_conditional_edges("OverallSupervisor", lambda x: x["next"],
                                            {
                                                "ResearchTeam": "ResearchTeam",
                                                "DocumentAuthoringTeam": "DocumentAuthoringTeam",
                                                "FINISH": END
                                            })
financial_audit_garph.add_edge(START, "OverallSupervisor")

In [ ]:
# Start audit
audit_results = financial_audit_garph.invoke({"company_id": "1234", "period": "Q4"})
print(audit_results)